# DeepNoise: End-to-End Pipeline

Welcome to the master pipeline notebook for the **DeepNoise Project**. This notebook coordinates the entire pipeline from data ingestion to feature engineering, model training, and model evaluation.

### Pipeline Structure
The workflow consists of 5 main stages:
1. **`download_data.py`**: Downloads raw acoustic WAV files from Zenodo.
2. **`preprocess.py`**: Normalizes sample rates, volume amplitude, and pads/truncates files to a standard 4.0 seconds.
3. **`extract_features.py`**: Extracts 2D Mel-spectrograms from the preprocessed audio and saves them as binary NumPy files.
4. **`train.py`**: Trains the custom 2D CNN in native PyTorch with early stopping and class-weighting, saving the best weights.
5. **`evaluate.py`**: Evaluates the trained model on unseen test data, printing classification reports and saving a confusion matrix heatmap.

---

## Step 1: Ingest Raw Dataset

We start by running the data downloader script to fetch the DCASE 2020 machine sound subsets (fan and pump) from Zenodo. The script downloads the zip archives directly, extracts them to `data/raw/`, and deletes the archives afterwards.

In [1]:
%run download_data.py


=== Processing Class: FAN ===
Extracted directory already exists for fan at: /home/jantofp/Documents/DeepNoiseProject/data/raw/fan. Skipping.

=== Processing Class: PUMP ===
Extracted directory already exists for pump at: /home/jantofp/Documents/DeepNoiseProject/data/raw/pump. Skipping.


## Step 2: Audio Preprocessing

Next, we run the preprocessor. This script standardizes the raw WAV files by:
- Resampling all audio to 22,050 Hz.
- Standardizing duration to 4.0 seconds (padding shorter clips with silence, truncating longer ones).
- Normalizing amplitude levels to a range of [-1.0, 1.0].

Processed waveforms are saved in parallel to `data/processed/` using your CPU cores.

In [2]:
%run preprocess.py

Scanning raw dataset directories...
Collected 9755 audio files for processing.

Starting parallel audio preprocessing...
Progress: 500/9755 files processed (5.1%) | Successes: 500 | Failures: 0
Progress: 1000/9755 files processed (10.3%) | Successes: 1000 | Failures: 0
Progress: 1500/9755 files processed (15.4%) | Successes: 1500 | Failures: 0
Progress: 2000/9755 files processed (20.5%) | Successes: 2000 | Failures: 0
Progress: 2500/9755 files processed (25.6%) | Successes: 2500 | Failures: 0
Progress: 3000/9755 files processed (30.8%) | Successes: 3000 | Failures: 0
Progress: 3500/9755 files processed (35.9%) | Successes: 3500 | Failures: 0
Progress: 4000/9755 files processed (41.0%) | Successes: 4000 | Failures: 0
Progress: 4500/9755 files processed (46.1%) | Successes: 4500 | Failures: 0
Progress: 5000/9755 files processed (51.3%) | Successes: 5000 | Failures: 0
Progress: 5500/9755 files processed (56.4%) | Successes: 5500 | Failures: 0
Progress: 6000/9755 files processed (61.5%) | 

## Step 3: Feature Extraction (Mel-Spectrograms)

With the standardized WAV files ready, we compute their 2D Mel-spectrogram frequency representations (128 bands by 173 time bins). These are scaled logarithmically (in decibels) and saved as binary NumPy `.npy` files under `data/features/` in parallel.

In [3]:
%run extract_features.py

Scanning processed audio directories...
Collected 9755 processed files for feature extraction.

Starting parallel Mel-spectrogram extraction...
Progress: 500/9755 features extracted (5.1%) | Successes: 500 | Failures: 0
Progress: 1000/9755 features extracted (10.3%) | Successes: 1000 | Failures: 0
Progress: 1500/9755 features extracted (15.4%) | Successes: 1500 | Failures: 0
Progress: 2000/9755 features extracted (20.5%) | Successes: 2000 | Failures: 0
Progress: 2500/9755 features extracted (25.6%) | Successes: 2500 | Failures: 0
Progress: 3000/9755 features extracted (30.8%) | Successes: 3000 | Failures: 0
Progress: 3500/9755 features extracted (35.9%) | Successes: 3500 | Failures: 0
Progress: 4000/9755 features extracted (41.0%) | Successes: 4000 | Failures: 0
Progress: 4500/9755 features extracted (46.1%) | Successes: 4500 | Failures: 0
Progress: 5000/9755 features extracted (51.3%) | Successes: 5000 | Failures: 0
Progress: 5500/9755 features extracted (56.4%) | Successes: 5500 | Fa

## Step 4: CNN Training (Native PyTorch)

Now we train our custom 2D CNN model using the extracted Mel-spectrogram features. The script:
- Loads features directly into system memory.
- Splits the dataset into 70% Train, 15% Validation, and 15% Test sets.
- Runs training on the target device (GPU if available).
- Saves the best model checkpoint to `models/best_cnn.pth` and the unseen test set to `models/test_split.npz`.

In [4]:
%run train.py

Scanning feature directories...
Class-to-Label mapping: {'fan_anomaly': 0, 'fan_normal': 1, 'pump_anomaly': 2, 'pump_normal': 3}
Loading 1475 files for class: fan_anomaly
Loading 4075 files for class: fan_normal
Loading 456 files for class: pump_anomaly
Loading 3749 files for class: pump_normal
Loaded dataset shapes: Features = (9755, 128, 173, 1) | Labels = (9755,)

Class Weights:
  Class 'fan_anomaly' (0): weight = 1.6534
  Class 'fan_normal' (1): weight = 0.5985
  Class 'pump_anomaly' (2): weight = 5.3481
  Class 'pump_normal' (3): weight = 0.6505

Building CNN model targeting device: cuda

Starting model training on GPU in native PyTorch (Epochs = 30, Batch Size = 16)...
Epoch 01/30 - loss: 1.3316 - accuracy: 0.4527 - val_loss: 1.5002 - val_accuracy: 0.3855
  val_loss improved, saving model checkpoint to: /home/jantofp/Documents/DeepNoiseProject/models/best_cnn.pth
Epoch 02/30 - loss: 1.1898 - accuracy: 0.5571 - val_loss: 1.1955 - val_accuracy: 0.6186
  val_loss improved, saving mo

## Step 5: Model Evaluation

Finally, we run the evaluation script to test the model's accuracy, precision, recall, and F1-score on the unseen test split. The script prints a classification report and saves a Confusion Matrix plot to `results/cnn_confusion_matrix.png`.

In [5]:
%run evaluate.py

Loading test split from: /home/jantofp/Documents/DeepNoiseProject/models/test_split.npz
Test Set Shapes: Features = (1464, 128, 173, 1) | Labels = (1464,)
Classes mapped to labels: {'fan_anomaly': 0, 'fan_normal': 1, 'pump_anomaly': 2, 'pump_normal': 3}
Instantiating model and loading weights from: /home/jantofp/Documents/DeepNoiseProject/models/best_cnn.pth
Running inference on cuda...

               CLASSIFICATION REPORT
              precision    recall  f1-score   support

 fan_anomaly       0.35      0.33      0.34       221
  fan_normal       0.76      0.80      0.78       612
pump_anomaly       0.26      0.18      0.21        68
 pump_normal       0.90      0.90      0.90       563

    accuracy                           0.74      1464
   macro avg       0.57      0.55      0.56      1464
weighted avg       0.73      0.74      0.73      1464

Generating Confusion Matrix...
Confusion Matrix saved to: /home/jantofp/Documents/DeepNoiseProject/results/cnn_confusion_matrix.png
